In [ ]:
import numpy as np
import yaml
from pathlib import Path
import batoid
from batoid_rubin import LSSTBuilder
from galsim.zernike import zernikeBasis
from tqdm import tqdm

In [ ]:
directory = Path("/Users/jmeyers3/src/ts_config_mttcs/MTAOS/v13/ofc/")
senspath = directory / "sensitivity_matrix" / "lsst_sensitivity_dz_31_29_50.yaml"
with open(senspath, "r") as f:
    mtaos_sens = np.array(yaml.safe_load(f))
mtaos_sens.shape

In [ ]:
thxs, thys = batoid.utils.hexapolar(outer=1.75, nrad=10, naz=62)

In [ ]:
fiducial = batoid.Optic.fromYaml("LSST_r.yaml")

In [ ]:
builder = LSSTBuilder(
    fiducial,
    dof_coord_system="OCS",
    flip_m2_bending_modes=False,
    dof_angle_units="degree"
)

tel0 = builder.build()
zk0 = []
for thx, thy in zip(thxs, thys):
    zk0.append(
        batoid.zernikeGQ(
            tel0,
            *np.deg2rad([thx, thy]),
            wavelength=622e-9,
            jmax=28,
            eps=0.62,
            rings=10
        ) * 0.622
    )
zk0 = np.array(zk0)

steps = [10, 100, 100, 0.001, 0.001]*2
steps += [0.1]*40

sens = []
for i, step in enumerate(tqdm(steps)):
    dof = [0]*50
    dof[i] = step
    builder = builder.with_aos_dof(dof)
    tel1 = builder.build()
    zk1 = []
    for thx, thy in zip(thxs, thys):
        zk1.append(
            batoid.zernikeGQ(
                tel1,
                *np.deg2rad([thx, thy]),
                wavelength=622e-9,
                jmax=28,
                eps=0.62,
                rings=10
            ) * 0.622
        )
    zk1 = np.array(zk1)
    sens.append((zk1 - zk0)/step)
sens = np.array(sens).transpose(0, 1, 2)

In [ ]:
# Now fit double Zernikes to the sens data
basis = zernikeBasis(30, thxs, thys, R_outer=1.75)
coefs, *_ = np.linalg.lstsq(
    basis.T,
    np.array(sens).transpose(1, 2, 0).reshape(len(thxs), -1),
    rcond=None
)
coefs = coefs.reshape(31, 29, 50)
coefs.shape

In [ ]:
import matplotlib.pyplot as plt
import ipywidgets

In [ ]:
mtaos_sens[0, ...] = 0
mtaos_sens[:, 0:4, :] = 0
coefs[0, ...] = 0
coefs[:, 0:4, :] = 0

In [ ]:
@ipywidgets.interact(i=(0, 49))
def f(i):
    vmin, vmax = np.min(coefs[..., i]), np.max(coefs[..., i])
    dmax = np.max(np.abs([vmin, vmax]))/10
    fig, axs = plt.subplots(nrows=1, ncols=3, figsize=(15, 5))
    axs[0].imshow(coefs[..., i], vmin=vmin, vmax=vmax)
    axs[0].set_title("Coefs")
    axs[1].imshow(-mtaos_sens[..., i], vmin=vmin, vmax=vmax)
    axs[1].set_title("MTAOS Sens")
    axs[2].imshow(coefs[..., i] + mtaos_sens[..., i], vmin=-dmax, vmax=dmax, cmap="bwr")
    axs[2].set_title("Diff")
    colorbar1 = fig.colorbar(axs[0].imshow(coefs[..., i], vmin=vmin, vmax=vmax), ax=axs, orientation='vertical')
    colorbar3 = fig.colorbar(axs[2].imshow(coefs[..., i] + mtaos_sens[..., i], vmin=-dmax, vmax=dmax, cmap="bwr"), ax=axs, orientation='vertical')

    plt.show()